# Classification NBA Model

## Configuration

## Imports

In [1]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    confusion_matrix,
    f1_score,
    make_scorer,
    precision_score,
    recall_score,
)
from nba_ou.data_preparation.missing_data.clean_df_for_training import (
    clean_dataframe_for_training
)
from sklearn.model_selection import KFold, cross_validate, train_test_split
from xgboost import XGBClassifier


In [2]:
from nba_ou.modeling.modeling import (
    split_latest_dates_holdout,
    make_walk_forward_last_n_seasons_splits,
    validate_time_splits,
    make_test_anchored_walk_forward_splits,
    assert_valid_time_splits,
    save_model_bundle,
    load_model_bundle
)

In [3]:
nan_threshold = 50.0
max_na_per_row = 5

## Load Data

In [4]:
data_path = "/home/adrian_alvarez/Projects/NBA_over_under_predictor/data/train_data/"
name = "all_odds_training_data_until_20260316.csv"

path = data_path + name

df_stats = pd.read_csv(path)

dtype_dict = {col: str for col in df_stats.columns if "ID" in col.upper()}

df_stats = pd.read_csv(
    path,
    dtype=dtype_dict
)
df_stats['GAME_DATE'] = pd.to_datetime(df_stats['GAME_DATE']).dt.strftime('%Y-%m-%d')

In [5]:
df_to_train = clean_dataframe_for_training(df_stats, nan_threshold=nan_threshold, max_na_per_row=max_na_per_row, create_missing_flags=False, verbose=1, keep_columns=['GAME_DATE'])

STARTING DATAFRAME CLEANING PIPELINE
Starting basic cleaning with 11241 rows
Basic cleaning complete: 8632 rows remaining

Starting advanced column cleaning with 3240 columns

Advanced column cleaning complete: 3240 → 2137 columns (1103 removed)


Dropping NA rows for SEASON_YEAR 2017...
   Removed 0 rows with NaN values from 2017 season

Applying missing data policy...

Missing Data Policy Report:
  Rows dropped: 0 (0.0%)
  Critical columns requiring data: 4
  Columns zero-filled: 112
  Infer pairs applied: 20/136
  Remaining NaN cells: 944087

Dropping rows with more than 5 NaN values...
Removed 3945 rows exceeding NaN threshold
CLEANING COMPLETE
Final shape: (4687, 2137)


In [6]:
# Count NAs per column
na_counts = df_to_train.isna().sum()

# Get most common SEASON_YEAR for nulls in each column
most_common_season = []
for col in df_to_train.columns:
    if na_counts[col] > 0:
        # Get rows where this column is null
        null_rows = df_to_train[df_to_train[col].isna()]
        if len(null_rows) > 0 and 'SEASON_YEAR' in df_to_train.columns:
            # Find most common SEASON_YEAR for these null rows
            common_season = null_rows['SEASON_YEAR'].mode()
            most_common_season.append(common_season.iloc[0] if len(common_season) > 0 else None)
        else:
            most_common_season.append(None)
    else:
        most_common_season.append(None)

na_counts_df = pd.DataFrame({
    'Column': na_counts.index,
    'NA_Count': na_counts.values,
    'NA_Percentage': (na_counts.values / len(df_to_train) * 100).round(2),
    'Most_Common_Season_Year': most_common_season
}).sort_values('NA_Count', ascending=False)

# Show only columns with NAs
na_counts_df[na_counts_df['NA_Count'] > 0]

,Column,NA_Count,NA_Percentage,Most_Common_Season_Year
1946,total_consensus_pct_under,67,1.43,2024.0
1947,spread_consensus_pct_home,63,1.34,2024.0
1953,ml_consensus_opener_price_away,14,0.30,2025.0
1954,ml_consensus_opener_price_home,14,0.30,2025.0
1801,total_consensus_pct_over_TREND_SLOPE_LAST_5_HO...,9,0.19,2023.0
1807,spread_consensus_pct_home_TREND_SLOPE_LAST_5_H...,9,0.19,2023.0
1805,spread_consensus_pct_away_TREND_SLOPE_LAST_5_H...,9,0.19,2023.0
1803,total_consensus_pct_under_TREND_SLOPE_LAST_5_H...,8,0.17,2023.0
1223,spread_consensus_pct_away_TREND_SLOPE_LAST_5_H...,5,0.11,2023.0
556,total_consensus_pct_over_TREND_SLOPE_LAST_5_HO...,5,0.11,2023.0


In [7]:
BET365_LINE_COL =  "TOTAL_LINE_bet365"
# BET365_LINE_COL =  "total_bet365_line_over"

# Ensure scoring line and target exist (avoid NaN-driven undefined betting accuracy).
df_to_train = df_to_train.dropna(subset=[BET365_LINE_COL, "TOTAL_POINTS"]).copy()

In [8]:
df_to_train['GAME_DATE'] = pd.to_datetime(df_to_train['GAME_DATE'])
df_to_train = df_to_train.sort_values("GAME_DATE").reset_index(drop=True)

In [9]:
#count games per season
games_per_season = df_to_train.groupby('SEASON_YEAR').size()
print(games_per_season)

SEASON_YEAR
2021    1158
2022    1173
2023     170
2024    1248
2025     938
dtype: int64


## Train / Test

In [10]:
from sklearn.model_selection import KFold, cross_validate, cross_val_predict, TimeSeriesSplit
from sklearn.metrics import mean_squared_error, mean_absolute_error, make_scorer, root_mean_squared_error
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import LinearRegression
from xgboost import XGBRegressor
from sklearn.base import clone

In [11]:
df_dev, df_test_final = split_latest_dates_holdout(
    df=df_to_train,
    date_col="GAME_DATE",
    test_size=0.05,
)

print(f"Development set size: {len(df_dev)}")
print(f"Final test set size: {len(df_test_final)}")
print("Final test date range:",
      df_test_final["GAME_DATE"].min(), "->", df_test_final["GAME_DATE"].max())

Development set size: 4452
Final test set size: 235
Final test date range: 2026-02-08 00:00:00 -> 2026-03-16 00:00:00


In [12]:
TARGET_COL = "TOTAL_POINTS"

EXCLUDE_COLS = [
    "TOTAL_POINTS",
    "SEASON_YEAR",
    "GAME_DATE",
]

X_dev = df_dev.drop(columns=EXCLUDE_COLS, errors="ignore")
y_dev = pd.to_numeric(df_dev[TARGET_COL], errors="coerce")

X_test_final = df_test_final.drop(columns=EXCLUDE_COLS, errors="ignore")
y_test_final = pd.to_numeric(df_test_final[TARGET_COL], errors="coerce")

print(f"X_dev shape: {X_dev.shape}")
print(f"X_test_final shape: {X_test_final.shape}")

X_dev shape: (4452, 2134)
X_test_final shape: (235, 2134)


In [13]:
from nba_ou.modeling.scorers import OverUnderScorerTotalPoints, over_under_betting_accuracy_total_points, evaluate_total_points_thresholds
    
ou_scorer = OverUnderScorerTotalPoints(BET365_LINE_COL)

scoring = {
    "MSE": "neg_mean_squared_error",
    "RMSE": "neg_root_mean_squared_error",
    "MAE": "neg_mean_absolute_error",
    "R2": "r2",
    "OU_Betting_Accuracy": ou_scorer,
}

def print_metrics(cv_results):
    for sc in scoring.keys():
        train_key = f"train_{sc}"
        test_key = f"test_{sc}"

        train_val = cv_results[train_key].mean()
        test_val = cv_results[test_key].mean()

        if sc in {"MSE", "RMSE", "MAE"}:
            train_val = -train_val
            test_val = -test_val

        if sc == "OU_Betting_Accuracy":
            print(f"Train {sc}: {train_val:.2%}")
            print(f"Validation {sc}: {test_val:.2%}")
        else:
            print(f"Train {sc}: {train_val:.5f}")
            print(f"Validation {sc}: {test_val:.5f}")
        print()

In [14]:
splits, fold_info = make_test_anchored_walk_forward_splits(
    df=df_dev,
    date_col="GAME_DATE",
    season_col="SEASON_YEAR",
    test_games=30,
    step_games_between_tests=100,
    train_games=3500,
    min_train_games=2000,
    max_folds=12,
    verbose=1,
)


Created 12 test-anchored walk-forward folds
 fold  train_n_games  test_n_games train_start_date train_end_date test_start_date test_end_date  test_season
    1           2776            35       2021-10-29     2024-12-07      2024-12-08    2024-12-15         2024
    2           2913            32       2021-10-29     2024-12-31      2025-01-01    2025-01-04         2024
    3           3046            37       2021-10-29     2025-01-18      2025-01-19    2025-01-23         2024
    4           3188            31       2021-10-29     2025-02-06      2025-02-07    2025-02-10         2024
    5           3322            33       2021-10-29     2025-03-01      2025-03-02    2025-03-05         2024
    6           3455            31       2021-10-29     2025-03-18      2025-03-19    2025-03-22         2024
    7           3500            38       2021-11-13     2025-04-04      2025-04-05    2025-04-09         2024
    8           3500            32       2021-12-06     2025-06-22      2025

In [15]:
season_bl = DummyRegressor(strategy="mean")

cv_results = cross_validate(
    season_bl,
    X_dev,
    y_dev,
    cv=splits,
    scoring=scoring,
    return_train_score=True,
    n_jobs=1,
)

print("DummyRegressor baseline")
print_metrics(cv_results)

DummyRegressor baseline
Train MSE: 381.17752
Validation MSE: 358.38897

Train RMSE: 19.52349
Validation RMSE: 18.87943

Train MAE: 15.58019
Validation MAE: 14.84988

Train R2: 0.00000
Validation R2: -0.09992

Train OU_Betting_Accuracy: 49.55%
Validation OU_Betting_Accuracy: 48.35%



In [16]:
lr = LinearRegression()

cv_results = cross_validate(
    lr,
    X_dev.fillna(0),   # LR cannot handle NaNs
    y_dev,
    cv=splits,
    scoring=scoring,
    return_train_score=True,
    n_jobs=-1,
)

print("Linear Regression")
print_metrics(cv_results)

Linear Regression
Train MSE: 160.29673
Validation MSE: 48915823.02134

Train RMSE: 12.65218
Validation RMSE: 2059.29769

Train MAE: 9.82761
Validation MAE: 1954.38787

Train R2: 0.57919
Validation R2: -178599.48385

Train OU_Betting_Accuracy: 73.79%
Validation OU_Betting_Accuracy: 46.11%



In [32]:
xgb_reg = XGBRegressor(
    max_depth=3,
    learning_rate=0.054,
    n_estimators=116,
    subsample=0.7,
    colsample_bytree=0.78,
    reg_alpha=7,
    reg_lambda=0.2,
    min_child_weight=13.23,
    gamma=1.82,
    n_jobs=-2,          # choose your CPU count
    random_state=16,
)

cv_results = cross_validate(
    xgb_reg,
    X_dev,
    y_dev,
    cv=splits,
    scoring=scoring,
    return_train_score=True,
    n_jobs=1,          # leave at 1 because XGB already parallelizes internally
)

print("XGBoost")
print_metrics(cv_results)

XGBoost
Train MSE: 209.77561
Validation MSE: 271.86796

Train RMSE: 14.48127
Validation RMSE: 16.43870

Train MAE: 11.47705
Validation MAE: 13.19125

Train R2: 0.44942
Validation R2: 0.15519

Train OU_Betting_Accuracy: 75.61%
Validation OU_Betting_Accuracy: 51.89%



In [34]:
xgb_reg.fit(X_dev, y_dev)

y_pred_test_total = xgb_reg.predict(X_test_final)

mse = mean_squared_error(y_test_final, y_pred_test_total)
rmse = root_mean_squared_error(y_test_final, y_pred_test_total)
mae = mean_absolute_error(y_test_final, y_pred_test_total)

betting_line = X_test_final[BET365_LINE_COL].to_numpy(dtype=float)

ou_acc = over_under_betting_accuracy_total_points(
    y_test_final,
    y_pred_test_total,
    betting_line,
)

print("Final test metrics")
print(f"MSE: {mse:.5f}")
print(f"RMSE: {rmse:.5f}")
print(f"MAE: {mae:.5f}")
print(f"OU_Betting_Accuracy: {ou_acc:.2%}")

Final test metrics
MSE: 302.86957
RMSE: 17.40315
MAE: 13.71507
OU_Betting_Accuracy: 51.53%


In [35]:
results_df, y_pred_test_total = evaluate_total_points_thresholds(
    model=xgb_reg,
    X_test=X_test_final,
    y_test_total=y_test_final,
    line_col=BET365_LINE_COL,
    thresholds=range(0, 11),
)

display(
    results_df.style.format(
        {"pct_of_test": "{:.1%}", "ou_betting_accuracy": "{:.2%}"}
    )
)

,threshold_abs_pred_edge_gt,n_games,pct_of_test,ou_betting_accuracy
0,0,235,100.0%,51.53%
1,1,168,71.5%,52.15%
2,2,104,44.3%,58.00%
3,3,57,24.3%,61.82%
4,4,29,12.3%,75.00%
5,5,10,4.3%,60.00%
6,6,4,1.7%,25.00%
7,7,3,1.3%,33.33%
8,8,0,0.0%,nan%
9,9,0,0.0%,nan%


# OPTUNA

In [20]:
from nba_ou.modeling.optuna_total_points import (
    tune_xgb_total_points_optuna,
    summarize_optuna_trials,
    fit_best_xgb_total_points,
)

study = tune_xgb_total_points_optuna(
    X=X_dev,
    y=y_dev,
    splits=splits,
    line_col=BET365_LINE_COL,
    n_trials=80,
    timeout=5.5 * 3600,
    objective_name="reg:squarederror",   # second run: reg:pseudohubererror
    study_name="xgb_total_points_mae",
)

print("Best CV MAE:", study.best_value)
print("Best params:")
for k, v in study.best_params.items():
    print(f"{k}: {v}")

print("\nSecondary metrics from best trial:")
print("Mean RMSE:", study.best_trial.user_attrs.get("mean_rmse"))
print("Mean R2:", study.best_trial.user_attrs.get("mean_r2"))
print("Mean OU accuracy:", study.best_trial.user_attrs.get("mean_ou_acc"))
print("Mean best_iteration:", study.best_trial.user_attrs.get("mean_best_iteration"))

trials_df = summarize_optuna_trials(study)
display(
    trials_df.head(15).style.format(
        {
            "value_mae": "{:.4f}",
            "mean_rmse": "{:.4f}",
            "mean_r2": "{:.4f}",
            "mean_ou_acc": "{:.2%}",
        }
    )
)

[I 2026-03-17 22:09:01,603] A new study created in memory with name: xgb_total_points_mae


  0%|          | 0/80 [00:00<?, ?it/s]

[I 2026-03-17 22:15:48,271] Trial 0 finished with value: 12.967673395078469 and parameters: {'max_depth': 2, 'min_child_weight': 14.839989016734002, 'gamma': 1.6521043697435434, 'subsample': 0.5682407800531226, 'colsample_bytree': 0.6123279759064977, 'learning_rate': 0.015902381379451283, 'reg_alpha': 1.8771791376898666, 'reg_lambda': 0.2766344129879233}. Best is trial 0 with value: 12.967673395078469.
[I 2026-03-17 22:24:00,075] Trial 1 finished with value: 12.93274574745658 and parameters: {'max_depth': 2, 'min_child_weight': 35.38241645406617, 'gamma': 1.6910441407044758, 'subsample': 0.5811969357559948, 'colsample_bytree': 0.7751882300061779, 'learning_rate': 0.013902617405829187, 'reg_alpha': 0.06701717253228541, 'reg_lambda': 0.6196026989845951}. Best is trial 1 with value: 12.93274574745658.
[I 2026-03-17 22:29:44,988] Trial 2 finished with value: 12.99282756224303 and parameters: {'max_depth': 4, 'min_child_weight': 13.12932060704323, 'gamma': 0.6451864311430185, 'subsample': 0

,trial,value_mae,mean_rmse,mean_r2,mean_ou_acc,mean_best_iteration,max_depth,min_child_weight,gamma,subsample,colsample_bytree,learning_rate,reg_alpha,reg_lambda
0,63,12.7097,16.2431,0.1774,54.29%,67,3,12.267528,1.437522,0.832049,0.814502,0.075640,5.003819,0.161371
1,65,12.7139,16.1989,0.1845,54.72%,75,3,12.171819,1.476818,0.793501,0.764929,0.078672,4.927710,0.119071
2,76,12.7243,16.1882,0.1835,57.05%,115,3,13.232517,1.825629,0.702550,0.786210,0.054892,7.097201,0.203288
3,66,12.7325,16.2025,0.1836,55.48%,80,3,12.103588,1.863127,0.810838,0.768301,0.079555,5.284879,0.100698
4,26,12.7489,16.2423,0.1789,54.54%,80,3,10.714055,0.548420,0.659017,0.677551,0.068018,0.019758,0.223616
5,30,12.7492,16.1370,0.1880,58.42%,126,4,16.336214,0.863615,0.849809,0.607265,0.051594,3.760079,0.302021
6,46,12.7573,16.2460,0.1785,55.51%,56,3,10.670495,1.421726,0.785587,0.814960,0.072331,7.349067,0.226143
7,51,12.7680,16.2710,0.1750,55.90%,78,3,10.582929,1.347809,0.830335,0.809223,0.061449,6.264488,0.256726
8,68,12.7736,16.1927,0.1825,55.49%,66,3,13.111256,1.862509,0.817385,0.741019,0.074740,2.264829,0.151232
9,73,12.7817,16.2384,0.1781,55.72%,77,3,12.548585,2.112539,0.823873,0.739354,0.075619,2.562637,0.199249


In [21]:
best_model = fit_best_xgb_total_points(
    X_dev=X_dev,
    y_dev=y_dev,
    study=study,
    objective_name="reg:squarederror",
)

y_pred_test_total = best_model.predict(X_test_final)

mse = mean_squared_error(y_test_final, y_pred_test_total)
rmse = root_mean_squared_error(y_test_final, y_pred_test_total)
mae = mean_absolute_error(y_test_final, y_pred_test_total)

betting_line = X_test_final[BET365_LINE_COL].to_numpy(dtype=float)

ou_acc = over_under_betting_accuracy_total_points(
    y_true=y_test_final,
    y_pred=y_pred_test_total,
    betting_line=betting_line,
)

print("Final test metrics")
print(f"MSE: {mse:.5f}")
print(f"RMSE: {rmse:.5f}")
print(f"MAE: {mae:.5f}")
print(f"OU_Betting_Accuracy: {ou_acc:.2%}")

Final test metrics
MSE: 304.33279
RMSE: 17.44514
MAE: 13.78343
OU_Betting_Accuracy: 49.78%


In [22]:
from nba_ou.modeling.modeling import ModelBundleMetadata, ModelInfo, TrainingMetrics

X_full = df_to_train.drop(columns=EXCLUDE_COLS, errors="ignore")
y_full = pd.to_numeric(df_to_train[TARGET_COL], errors="coerce")

production_model = fit_best_xgb_total_points(
    X_dev=X_full,
    y_dev=y_full,
    study=study,
    objective_name="reg:squarederror",
)

latest_training_date = pd.to_datetime(df_to_train["GAME_DATE"]).max()
model_version = latest_training_date.strftime("%d_%m_%y")
model_name = f"five_seasons_xgb_total_points_{model_version}"

metadata = ModelBundleMetadata(
    model_info=ModelInfo(
        name=model_name,
        model_version=model_version,
        model_type="five_seasons_total_points",
        prediction_source="five_seasons_xgb_total_points",
        training_code_tag="1.0",
    ),
    training_metrics=TrainingMetrics(
        best_params=study.best_trial.params,
        mean_best_iteration=study.best_trial.user_attrs.get("mean_best_iteration"),
        cv_mae=float(study.best_value),
        cv_rmse=study.best_trial.user_attrs.get("mean_rmse"),
        cv_ou_acc=study.best_trial.user_attrs.get("mean_ou_acc"),
        final_test_mae=float(mae),
        final_test_rmse=float(rmse),
        final_test_ou_acc=float(ou_acc),
        nan_threshold=nan_threshold,
        max_na_per_row=max_na_per_row,
        train_date_min=df_to_train["GAME_DATE"].min().to_pydatetime(),
        train_date_max=df_to_train["GAME_DATE"].max().to_pydatetime(),
    ),
)

model_path, meta_path = save_model_bundle(
    model=production_model,
    feature_names=list(X_full.columns),
    out_dir="/home/adrian_alvarez/Projects/NBA_over_under_predictor/models/total_points/5_seasons/",
    metadata=metadata,
)

print(f"Production model trained on {len(X_full)} rows using fixed n_estimators from mean_best_iteration.")
print("Saved model :", model_path)
print("Saved metadata:", meta_path)


Production model trained on 4687 rows using fixed n_estimators from mean_best_iteration.
Saved model : /home/adrian_alvarez/Projects/NBA_over_under_predictor/models/total_points/5_seasons/five_seasons_xgb_total_points_16_03_26.json
Saved metadata: /home/adrian_alvarez/Projects/NBA_over_under_predictor/models/total_points/5_seasons/five_seasons_xgb_total_points_16_03_26.meta.json


In [28]:
study.best_trial.user_attrs.get("mean_best_iteration")

67